# Inter-Eval Structural Analysis

**What:** Does the structural fingerprint of the reference (human) patch for a task predict whether an AI agent can solve it?

**Data provenance:**
- The *traces* are human patches — the ground-truth fixes from GitHub that closed each issue, pulled from the SWE-Bench HuggingFace dataset and parsed into before/after file diffs.
- The *pass/fail labels* come separately from agent run results (`output/swebench_results/`). Pass = the agent's generated patch made the test suite pass.
- All reference patches are correct by definition. "Passed" means a specific agent solved that task, not that the patch itself is valid.

| Agent | Tasks solved | Pass rate |
|---|---|---|
| SWE-Agent + GPT-4 (Apr 2024) | 54 / 300 | 18% |
| SWE-Agent + Claude 3.5 Sonnet (Jun 2024) | 69 / 300 | 23% |
| RAG + GPT-4 (Apr 2024) | 8 / 300 | 3% |

**Research question:** Do structural properties of the human reference patch — how many files changed, what edit operations were used, what co-edit patterns appear — predict which tasks an agent is likely to solve?

**Benchmarks compared:**
- `swe_bench_lite_resolved` — 300 instances, 12 repos, labeled by SWE-Agent + GPT-4
- `swe_smith_resolved` — 300 synthetic instances generated from the SWE-Bench methodology (no agent pass/fail labels)

Plots are precomputed by `scripts/run_cross_benchmark_analysis.py`. This notebook loads and displays them with a results summary.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

ROOT = Path("../").resolve()
DATA_DIR = ROOT / "output" / "datasets"
PLOTS_DIR = Path("plots/cross_benchmark").resolve()

DATASETS = [
    "swe_bench_lite_resolved",
    "swe_smith_resolved",
]

def show(filename: str, width: int = 700):
    p = PLOTS_DIR / filename
    if p.exists():
        display(Image(str(p), width=width))
    else:
        print(f"Missing: {filename} — run scripts/run_cross_benchmark_analysis.py first")

: 

## Results summary

Key numbers from precomputed outputs:

In [ ]:
import pandas as pd

rows = []
for ds in DATASETS:
    for p in sorted((DATA_DIR / ds).glob("transfer_metrics*.json")):
        d = json.loads(p.read_text())
        rows.append({
            "benchmark": ds.replace("swe_bench_", "").replace("_resolved", "").replace("_multifile", " (multifile)"),
            "repr": d["repr"],
            "n_passed": d["n_passed"],
            "pass_rate": f"{d['overall_pass_rate']:.0%}",
            "auc": round(d["auc_distance_vs_pass"], 3),
            "knn_k5": round(d["knn"]["k=5"]["accuracy"], 3) if d.get("knn") else None,
            "saturation_at": d["saturation_knee_rank"],
        })

summary = pd.DataFrame(rows)
summary

**Interpretation:**
- AUC ~0.45–0.48 for edits and modules on `lite_resolved` — near-random. Structural distance from human patches barely separates agent-solvable from unsolvable tasks on this representation.
- AUC = 1.0 for `motifs` on `lite_resolved` — **flag for data leakage**: motif vocabulary was likely built on this split, so it perfectly encodes which instances it was trained on.
- AUC = 0.60 for `edits_set_diff` on `verified_multifile` — stronger signal on the multifile subset. Multifile patches may be structurally more distinctive.
- Saturation knee ~15–18 across all representations: the structural signal (in terms of coverage) saturates after seeing ~16 examples, regardless of representation.
- Cross-benchmark transfer (lite → verified, 50 overlapping instances): **86% accuracy**, vs 84% source accuracy. The structural signal transfers cleanly.

## AUC heatmap: representation × benchmark

In [ ]:
show("cross_auc_heatmap.png")

## Saturation: how many examples until structural signal plateaus

In [ ]:
show("cross_saturation_bar.png")
show("cross_saturation_pct.png")

## Pass rates and regional breakdown

In [ ]:
show("cross_pass_rate.png")
show("cross_region_pass_rates.png")

## Fix type distributions

In [ ]:
show("cross_fix_types.png")

## Summary table

In [ ]:
show("cross_summary_table.png")

## Representation diversity: are the structural representations independent?

Each representation encodes a different structural facet — edit operations, module co-edits, recurring patterns. These two plots ask whether they actually carry different information.

**Unique variance** — how much of the total structural signal does each representation contribute that no other representation already captures?

**Rank correlation** — Spearman ρ between pairwise distance vectors. ρ = 1.0 means the two representations produce identical rankings of task similarity and are fully redundant.

In [ ]:
show("cross_diversity_unique_variance.png", width=560)
show("cross_diversity_rank_corr.png", width=640)

**Findings (consistent across Lite and SWE-Smith):**
- `dependency graph` (modules) and `recurring patterns` (motifs) each carry nearly all unique structural signal independently — both ~100% unique variance.
- `raw edits` and `edit set-diff` are essentially redundant: ρ = 1.0 with each other, near-zero unique variance on top of the other representations.
- `dependency graph` is orthogonal to all edit representations (ρ ≈ 0): file-level co-edit structure and operation-level edit structure are genuinely different axes.
- The two-dimensional structure (edits + modules) is stable across real and synthetic tasks.

---
## Next: extending to different eval types

All current benchmarks are SWE-Bench variants (patch-based bug fixes). The inter-eval comparison is currently intra-benchmark in nature.

Two additions that add genuine structural diversity:

1. **SWE-Smith** — already configured in `configs/benchmarks.yaml`. 50k synthetic patch tasks. Run with:
   ```
   uv run python scripts/run_benchmark_pipeline.py --benchmark swe_smith --limit 300
   uv run python scripts/run_cross_benchmark_analysis.py --benchmarks swe_bench_lite_resolved swe_smith_resolved
   ```

2. **MBPP** — function synthesis tasks (write a function from a docstring). Requires a new loader that converts blank-to-solution into the same trace format. The downstream pipeline (distance matrices, diversity, transfer metrics) reuses unchanged.